# 01 · Data preparation

**Purpose.** Load the CPCB city-day data, keep the 7 study cities, apply the data-quality rule
(AQI capped at 500), report missing values, and clean each city into a continuous daily series
ready for feature engineering.

**Inputs**
- `../data/city_day.csv`: CPCB monitoring data, daily, city level (2015-01-01 to 2020-07-01)

**Outputs**
- `../results/missing_values_pct.csv`: % missing per pollutant and city (raw data)
- `../data/processed/clean.csv`: cleaned daily series for the 7 cities (used by notebook 03)

## Configuration
Constants used in this notebook (same values as the original `src/config.py`). Paths are relative
to the `notebooks/` folder.

In [1]:
from pathlib import Path

DATA_PATH = Path("../data/city_day.csv")
PROCESSED_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results")

CITIES = ["Ahmedabad", "Bengaluru", "Chennai", "Delhi", "Hyderabad", "Lucknow", "Mumbai"]

POLLUTANTS = ["PM2.5", "PM10", "NO2", "SO2", "CO", "O3", "NH3"]

TARGET = "AQI"
AQI_SCALE_MAX = 500
FFILL_LIMIT = 3

In [2]:
import pandas as pd

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

## 1. Load the data and keep the 7 study cities
The dataset covers many cities, but only these 7 have near-complete daily coverage over
2015 to mid-2020, which a daily forecasting study needs.

In [3]:
df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
raw = df[df["City"].isin(CITIES)].copy()
print(f"All cities: {df.shape}   7 study cities: {raw.shape}")
raw.head()

All cities: (29531, 16)   7 study cities: (14060, 16)


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN,NaN


## 2. Data-quality rule: cap AQI at 500
Ahmedabad's CO readings are physically implausible (median about 16 mg/m³ against about 1.2 in
Delhi, with spikes above 100). Since CO enters the AQI formula, this pushes Ahmedabad's reported
AQI as high as 2,049. India's National AQI scale ends at 500, so values above it are clipped.
Results in notebook 04 are also reported with Ahmedabad excluded.

Maximum reported AQI per city **before** the cap:

In [4]:
raw.groupby("City")["AQI"].max()

City
Ahmedabad    2049.0
Bengaluru     352.0
Chennai       449.0
Delhi         716.0
Hyderabad     737.0
Lucknow       707.0
Mumbai        307.0
Name: AQI, dtype: float64

In [5]:
raw["AQI"] = raw["AQI"].clip(upper=AQI_SCALE_MAX)
print(f"Raw rows for the 7 study cities: {len(raw)}")
raw.groupby("City")["AQI"].max()

Raw rows for the 7 study cities: 14060


City
Ahmedabad    500.0
Bengaluru    352.0
Chennai      449.0
Delhi        500.0
Hyderabad    500.0
Lucknow      500.0
Mumbai       307.0
Name: AQI, dtype: float64

## 3. Missing-value report
Share of missing values (%) per pollutant and city in the raw data, before any cleaning.

In [6]:
missing = (raw.groupby("City")[POLLUTANTS + [TARGET]]
              .apply(lambda g: g.isna().mean().mul(100).round(1)))
missing.to_csv(RESULTS_DIR / "missing_values_pct.csv")
missing

,PM2.5,PM10,NO2,SO2,CO,O3,NH3,AQI
City,,,,,,,,
Ahmedabad,31.3,79.7,30.3,31.9,30.3,34.1,100.0,33.6
Bengaluru,7.3,17.9,0.3,0.3,0.5,7.2,10.1,4.9
Chennai,5.8,85.0,1.8,1.8,1.2,2.4,23.0,6.2
Delhi,0.1,3.8,0.1,5.5,0.0,4.2,0.4,0.5
Hyderabad,5.7,17.7,1.4,1.3,0.2,1.3,17.9,6.3
Lucknow,5.1,100.0,1.2,3.2,1.4,1.3,50.0,5.8
Mumbai,61.0,62.0,62.4,60.7,1.2,60.3,80.3,61.4


## 4. Cleaning
Each city is cleaned separately, so one city's values never fill another city's gaps:

- **Continuous daily calendar.** Each city is reindexed to every day between its first and last
  date, and missing dates become explicit NaNs. Then a shift of 1 in notebook 03 really means
  "yesterday", not "the previous row that happens to exist".
- **Negative concentrations → NaN.** They are physically impossible.
- **Forward fill, past only, at most 3 days.** Short input gaps are filled with the last observed
  value. Forward fill uses only past values. Linear interpolation would use the next observed
  value, i.e. information from the future, which leaks into a forecasting model. The 3-day limit
  keeps long outages as NaN instead of carrying one stale value for weeks.
- **`AQI_observed` keeps the original, unfilled AQI.** The target is never imputed. AQI is
  forward-filled only so its *lagged* values can serve as inputs. Rows without a measured AQI are
  dropped at train/test time, so a model is never trained or scored on a made-up target.

In [7]:
def clean_city(g: pd.DataFrame, city: str) -> pd.DataFrame:
    g = g.set_index("Date").sort_index()
    full_idx = pd.date_range(g.index.min(), g.index.max(), freq="D")
    g = g.reindex(full_idx)
    g["City"] = city
    g.index.name = "Date"

    cols = POLLUTANTS + [TARGET]
    g[cols] = g[cols].where(g[cols] >= 0)
    g["AQI_observed"] = g[TARGET]
    g[cols] = g[cols].ffill(limit=FFILL_LIMIT)
    return g.reset_index()

In [8]:
clean_df = pd.concat([clean_city(g, city) for city, g in raw.groupby("City")], ignore_index=True)
clean_df = clean_df[["City", "Date"] + POLLUTANTS + [TARGET, "AQI_observed"]]
print(clean_df.shape)
clean_df.head()

(14060, 11)


,City,Date,PM2.5,PM10,NO2,SO2,CO,O3,NH3,AQI,AQI_observed
0,Ahmedabad,2015-01-01,NaN,NaN,18.22,27.64,0.92,133.36,NaN,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,15.69,24.55,0.97,34.06,NaN,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,19.30,29.07,17.40,30.70,NaN,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,18.48,18.59,1.70,36.08,NaN,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,21.42,39.33,22.10,39.31,NaN,NaN,NaN


Rows per city after reindexing, plus how many days have a measured AQI (`AQI_observed`) versus an
AQI available after forward fill (`AQI`). The gap between the two is the filled days, which are
used only as lagged inputs, never as targets.

In [9]:
clean_df.groupby("City").agg(days=("Date", "size"), first=("Date", "min"), last=("Date", "max"),
                             AQI_observed=("AQI_observed", "count"), AQI_after_ffill=("AQI", "count"))

,days,first,last,AQI_observed,AQI_after_ffill
City,,,,,
Ahmedabad,2009,2015-01-01,2020-07-01,1334,1400
Bengaluru,2009,2015-01-01,2020-07-01,1910,1926
Chennai,2009,2015-01-01,2020-07-01,1884,1905
Delhi,2009,2015-01-01,2020-07-01,1999,2008
Hyderabad,2006,2015-01-04,2020-07-01,1880,1901
Lucknow,2009,2015-01-01,2020-07-01,1893,1914
Mumbai,2009,2015-01-01,2020-07-01,775,785


## 5. Save the cleaned data

In [10]:
clean_df.to_csv(PROCESSED_DIR / "clean.csv", index=False)
print("Saved", PROCESSED_DIR / "clean.csv", clean_df.shape)

Saved ..\data\processed\clean.csv (14060, 11)
